In [0]:
# ========================================================
# INGEST SUPABASE POSTGRESQL QUA SPARK JDBC (POOLER IPv4)
# Không dùng psycopg2 - Không lo treo Kernel - Không lỗi IPv6
# ========================================================

from pyspark.sql.functions import current_timestamp, lit

TEN = "khoa"

# 1. Cấu hình thông tin Supabase Connection Pooler
POOLER_HOST = "aws-0-ap-south-1.pooler.supabase.com"  # Thay đúng region pooler của bạn
POOLER_PORT = 5432                                       # Cổng Pooler Transaction mode
DB_NAME     = "postgres"
PROJECT_REF = "zrvziaxxaasagqizgvfd"                     # Mã project ref của bạn
DB_USER     = f"postgres.{PROJECT_REF}"                  # User bắt buộc có đuôi .project_ref
DB_PASS     = "KvGlwPjoh4WFgB9R"

# 2. Xây dựng JDBC URL an toàn với sslmode và prepareThreshold=0
# prepareThreshold=0 là chìa khóa để chạy mượt mà trên PgBouncer / Pooler
jdbc_url = (
    f"jdbc:postgresql://{POOLER_HOST}:{POOLER_PORT}/{DB_NAME}"
    f"?sslmode=require&prepareThreshold=0"
)

# 3. Đọc dữ liệu phân tán bằng Spark JDBC
print("Đang kết nối Supabase Pooler qua Spark JDBC...")
df_supabase = (
    spark.read.format("jdbc")
         .option("url", jdbc_url)
         .option("dbtable", "orders")
         .option("user", DB_USER)
         .option("password", DB_PASS)
         .option("driver", "org.postgresql.Driver")
         .load()
)

print(f"Đã đọc thành công {df_supabase.count()} bản ghi từ Supabase!")

# 4. Bổ sung metadata kỹ thuật và ghi vào Bronze Delta Table
TARGET_SUPABASE_TABLE = f"workspace.bronze_{TEN}.raw_orders_supabase"

(df_supabase
   .withColumn("_ingested_at", current_timestamp())
   .withColumn("_source_type", lit("SUPABASE_SPARK_JDBC_POOLER"))
   .write
   .format("delta")
   .mode("overwrite")
   .saveAsTable(TARGET_SUPABASE_TABLE))

print(f"Đã lưu dữ liệu vào bảng Bronze: {TARGET_SUPABASE_TABLE}")


In [0]:
# 1. Lấy mốc thời gian lớn nhất hiện có trong Bronze
try:
    watermark_row = spark.sql(f"""
        SELECT COALESCE(MAX(updated_at), '1970-01-01 00:00:00') AS max_ts 
        FROM workspace.bronze_{TEN}.raw_orders_supabase
    """).collect()[0]["max_ts"]
except Exception:
    watermark_row = "1970-01-01 00:00:00"

print("Watermark hiện tại:", watermark_row)

# 2. Sử dụng subquery (dbtable) trong Spark JDBC để chỉ kéo các dòng mới
cdc_query = f"(SELECT * FROM orders WHERE updated_at > '{watermark_row}') AS cdc_orders"

df_cdc = (
    spark.read.format("jdbc")
         .option("url", jdbc_url)
         .option("dbtable", cdc_query)
         .option("user", DB_USER)
         .option("password", DB_PASS)
         .option("driver", "org.postgresql.Driver")
         .load()
)

cdc_count = df_cdc.count()
print(f"Số bản ghi thay đổi mới phát hiện từ Supabase: {cdc_count}")

# 3. Thực hiện MERGE INTO bảng Bronze
if cdc_count > 0:
    df_cdc.createOrReplaceTempView("stg_supabase_cdc")
    spark.sql(f"""
        MERGE INTO workspace.bronze_{TEN}.raw_orders_supabase AS target
        USING stg_supabase_cdc AS source
        ON target.order_id = source.order_id
        WHEN MATCHED THEN
          UPDATE SET *
        WHEN NOT MATCHED THEN
          INSERT *
    """)
    print("Đã cập nhật đồng bộ CDC thành công vào Bronze!")


2. NGUỒN 2: BACKBLAZE B2 (S3-COMPATIBLE OBJECT STORAGE)

In [0]:
%pip install boto3

import boto3
import os

# 1. Thông số Backblaze B2 của bạn
B2_ENDPOINT    = "https://s3.us-east-005.backblazeb2.com"
B2_KEY_ID      = "0057a6c9cb8280e0000000001"
B2_APP_KEY     = "K005pwPN+jBLNxEG6tq6Vaa3pSppLu4"
B2_BUCKET      = "databricks-lab-khoa-2026"
VOLUME_LANDING = "/Volumes/workspace/lab_khoa/raw/landing"

# 2. Tạo Volume (nếu chưa có) và thư mục landing
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.lab_khoa")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.lab_khoa.raw")
dbutils.fs.mkdirs(VOLUME_LANDING)

# 3. Kết nối Backblaze B2 và kéo 3 file CSV về Volume
s3 = boto3.client(
    "s3",
    endpoint_url=B2_ENDPOINT,
    aws_access_key_id=B2_KEY_ID,
    aws_secret_access_key=B2_APP_KEY
)

response = s3.list_objects_v2(Bucket=B2_BUCKET)
for obj in response.get("Contents", []):
    key = obj["Key"]
    if not key.endswith("/"):
        dest = f"{VOLUME_LANDING}/{os.path.basename(key)}"
        s3.download_file(B2_BUCKET, key, dest)
        print(f"Đã kéo file {key} về {dest}")